# Biohub - Cell Tracking S1 Pipeline (StarDist 3D + btrack)

This notebook implements the **S1 Strategy Pipeline (CV Target: 0.90+)** for the **Biohub - Cell Tracking During Development** competition.

このノートブックは、**S1戦略 (CV目標: 0.90+)** を実現するための統合検証ノートブックです。

In [ ]:
import os
import sys
import time
import numpy as np
import pandas as pd
import zarr
import btrack
from skimage.feature import blob_dog
import tracksdata

# Add src modules to sys.path
sys.path.append(os.path.abspath("../src"))
from track.btrack_tracker import run_btrack_tracking

print("Imported libraries successfully!")
print(f"btrack version: {btrack.__version__}")

In [ ]:
# Path definitions
BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))
DATA_DIR = os.path.join(BASE_DIR, "data", "train")
KAGGLE_SRC_DIR = os.path.join(BASE_DIR, "src", "kaggle_cell_tracking_competition", "src", "tracking_cellmot")

if KAGGLE_SRC_DIR not in sys.path:
    sys.path.append(KAGGLE_SRC_DIR)

from evaluate import evaluate
from open_dataset import open_dataset

dataset_names = sorted([d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))])
print(f"Data Directory: {DATA_DIR}")
print(f"Found {len(dataset_names)} training datasets: {dataset_names}")

## 1. Node Detection & btrack Evaluation Function
## 1. ノード検出 ＋ btrack トラッキング評価関数の定義

In [ ]:
def detect_nodes_blob_dog(dataset_path, min_sigma=1.0, max_sigma=4.0, threshold=0.04):
    """
    Detect 3D cell centroids across time frames using blob_dog (Baseline Detection).
    """
    ds = open_dataset(dataset_path, normalize=True, require_tracks=False)
    images = ds.images  # shape: (T, Z, Y, X)
    n_frames = images.shape[0]
    
    nodes = []
    global_node_id = 0
    
    for t in range(n_frames):
        frame = images[t]
        # 3D blob detection
        blobs = blob_dog(frame, min_sigma=min_sigma, max_sigma=max_sigma, threshold=threshold)
        
        for blob in blobs:
            z, y, x, r = blob
            nodes.append({
                'node_id': global_node_id,
                't': t,
                'z': float(z),
                'y': float(y),
                'x': float(x)
            })
            global_node_id += 1
            
    return pd.DataFrame(nodes)

## 2. Run Validation on Dataset 0 with btrack (Bayesian + ILP)
## 2. 訓練データセット 0 における btrack トラッキング検証

In [ ]:
target_dataset_name = dataset_names[0]
target_dataset_path = os.path.join(DATA_DIR, target_dataset_name)
print(f"--- Evaluating Dataset: {target_dataset_name} ---")

# 1. Load Ground Truth Graph
ds_gt = open_dataset(target_dataset_path, normalize=True, require_tracks=True)
gt_graph = ds_gt.tracks
scale = ds_gt.scale

# 2. Node Detection
print("Running 3D node detection...")
nodes_df = detect_nodes_blob_dog(target_dataset_path)
print(f"Detected {len(nodes_df)} total nodes.")

# 3. Run btrack Tracking
print("Running btrack (Bayesian + ILP Global Tracking)...")
start_time = time.time()
edges = run_btrack_tracking(nodes_df, max_search_radius=25.0)
elapsed = time.time() - start_time
print(f"btrack generated {len(edges)} edges in {elapsed:.2f} seconds.")

# 4. Build TracksData Graph for Kaggle Evaluation
pred_graph = tracksdata.TracksData()
for row in nodes_df.itertuples():
    pred_graph.add_node(int(row.node_id), t=int(row.t), z=float(row.z), y=float(row.y), x=float(row.x))

for edge in edges:
    pred_graph.add_edge(int(edge['source_id']), int(edge['target_id']))

# 5. Calculate Score with Kaggle Evaluation Metric
eval_res = evaluate(graph=pred_graph, gt_graph=gt_graph, scale=scale)

edge_denom = eval_res.edge_tp + eval_res.edge_fp + eval_res.edge_fn
edge_jaccard = eval_res.edge_tp / edge_denom if edge_denom > 0 else 1.0

div_denom = eval_res.division_tp + eval_res.division_fp + eval_res.division_fn
div_jaccard = eval_res.division_tp / div_denom if div_denom > 0 else 1.0

final_score = edge_jaccard + 0.1 * div_jaccard

print("\n===== CV EVALUATION RESULTS (btrack) =====")
print(f"Edge TP: {eval_res.edge_tp}, FP: {eval_res.edge_fp}, FN: {eval_res.edge_fn}")
print(f"Edge Jaccard Score:     {edge_jaccard:.4f}")
print(f"Division Jaccard Score: {div_jaccard:.4f}")
print(f"OVERALL CV SCORE:       {final_score:.4f}")
print("===========================================")